# 面试题：HNSW 为什么快，插入、搜索、删除和参数怎样实现？

本 Notebook 用 NumPy 和 Python heap 手写一个教学版 HNSW：指数层级、顶层贪心导航、底层 `ef` 候选搜索、双向邻接、启发式简化选邻、度数裁剪、插入、查询、软删除、Recall@K 和图快照。

它不追求 hnswlib 的锁、SIMD、压缩和并发性能，但每个状态转换都可检查，适合面试解释 `M/efConstruction/efSearch` 的作用。

In [ ]:
import copy,hashlib,heapq,json,math,warnings  # 导入本单元所需的依赖。
from collections import defaultdict  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore",message="The pynvml package is deprecated")  # 计算并保存当前步骤的中间状态。
import numpy as np  # 导入本单元所需的依赖。
RNG69=np.random.default_rng(6901)  # 计算并保存当前步骤的中间状态。
def canonical69(x): return json.dumps(x,sort_keys=True,separators=(",",":"))  # 定义本节可复用的核心函数。
def sha69(x): return hashlib.sha256(x).hexdigest()  # 定义本节可复用的核心函数。
assert RNG69 is not None  # 用受控断言验证关键不变量。

## 1. 距离与数据合同

使用 6 个二维簇构造 train/index/query。索引向量必须定长、float32、有限，ID 唯一。HNSW 图结构只对固定距离有意义；若 embedding 或 normalization 升级，不能把新旧向量混在同一图中。

本例使用平方 L2，省掉开方且保持排序。生产中 cosine 常通过单位归一化转为内积或 L2。

In [ ]:
centers69=np.array([[-4,-4],[-4,0],[-4,4],[4,-4],[4,0],[4,4]],np.float32)  # 计算并保存当前步骤的中间状态。
data69=np.vstack([c+RNG69.normal(scale=.65,size=(35,2)) for c in centers69]).astype(np.float32)  # 计算并保存当前步骤的中间状态。
query69=np.vstack([c+RNG69.normal(scale=.35,size=(4,2)) for c in centers69]).astype(np.float32)  # 计算并保存当前步骤的中间状态。
ids69=[f"p{i:03d}" for i in range(len(data69))]  # 计算并保存当前步骤的中间状态。
def dist69(a,b):  # 定义本节可复用的核心函数。
    a=np.asarray(a,np.float32); b=np.asarray(b,np.float32)  # 计算并保存当前步骤的中间状态。
    if a.shape!=b.shape or a.ndim!=1 or not np.isfinite(a).all() or not np.isfinite(b).all(): raise ValueError("distance_contract")  # 按当前条件选择后续控制路径。
    return float(np.dot(a-b,a-b))  # 返回当前分支计算出的结果。
assert data69.shape==(210,2) and query69.shape==(24,2) and len(set(ids69))==210  # 用受控断言验证关键不变量。
assert dist69(data69[0],data69[0])==0 and dist69(data69[0],data69[1])>=0  # 用受控断言验证关键不变量。
try: dist69(np.zeros(2),np.zeros(3)); raise AssertionError("wrong distance shape accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="distance_contract"  # 捕获预期异常并验证失败分支。

## 2. 随机层级与可复现性

每个节点 level 服从几何/指数尾部：绝大多数只在第 0 层，少数进入高层形成高速公路。这里反复抛硬币，概率 `1/M` 升一层，并设最大层防止极端值。

level 是持久化图结构的一部分，必须由显式随机源产生。查询本身不随机；同一快照和参数应逐位复现。

In [ ]:
class LevelSampler69:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,seed,up_probability=.25,max_level=8): self.rng=np.random.default_rng(seed); self.p=up_probability; self.max=max_level  # 定义本节可复用的核心函数。
    def sample(self):  # 定义本节可复用的核心函数。
        level=0  # 计算并保存当前步骤的中间状态。
        while level<self.max and self.rng.random()<self.p: level+=1  # 在终止条件满足前持续推进状态。
        return level  # 返回当前分支计算出的结果。
levels_a69=[LevelSampler69(3).sample() for _ in range(3)]  # 计算并保存当前步骤的中间状态。
sampler_a69=LevelSampler69(9); seq_a69=[sampler_a69.sample() for _ in range(100)]  # 计算并保存当前步骤的中间状态。
sampler_b69=LevelSampler69(9); seq_b69=[sampler_b69.sample() for _ in range(100)]  # 计算并保存当前步骤的中间状态。
assert seq_a69==seq_b69 and max(seq_a69)<=8 and sum(x==0 for x in seq_a69)>50  # 用受控断言验证关键不变量。
assert all(isinstance(x,int) and x>=0 for x in seq_a69)  # 用受控断言验证关键不变量。

## 3. 单层贪心与 `ef` best-first 搜索

`greedy` 从 entry 反复走向更近邻居，适合高层导航。到目标层后，`search_layer` 同时维护待扩展最小堆和当前最佳最大堆；当最近待扩展点都比当前最差结果更远时停止。

`ef` 是搜索宽度而非返回数。`ef>=k`，增大通常提高召回但增加距离计算。删除节点不能作为结果，但仍可作为导航桥，直到重建图。

In [ ]:
class HNSW69:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,m=6,ef_construction=24,seed=6902):  # 定义本节可复用的核心函数。
        self.m=m; self.ef_construction=ef_construction; self.sampler=LevelSampler69(seed,1/max(m,2)); self.vectors={}; self.levels={}; self.graph=defaultdict(lambda:defaultdict(set)); self.entry=None; self.max_level=-1; self.deleted=set()  # 计算并保存当前步骤的中间状态。
    def _d(self,q,node): return dist69(q,self.vectors[node])  # 定义本节可复用的核心函数。
    def greedy(self,q,entry,level):  # 定义本节可复用的核心函数。
        current=entry; best=self._d(q,current); changed=True  # 计算并保存当前步骤的中间状态。
        while changed:  # 在终止条件满足前持续推进状态。
            changed=False  # 计算并保存当前步骤的中间状态。
            for other in sorted(self.graph[level].get(current,set())):  # 遍历输入元素以累积或检查结果。
                d=self._d(q,other)  # 计算并保存当前步骤的中间状态。
                if d<best: current,best,changed=other,d,True  # 按当前条件选择后续控制路径。
        return current  # 返回当前分支计算出的结果。
    def search_layer(self,q,entries,ef,level):  # 定义本节可复用的核心函数。
        visited=set(entries); candidates=[(self._d(q,e),e) for e in entries]; heapq.heapify(candidates)  # 计算并保存当前步骤的中间状态。
        best=[(-self._d(q,e),e) for e in entries]; heapq.heapify(best)  # 计算并保存当前步骤的中间状态。
        while candidates:  # 在终止条件满足前持续推进状态。
            d,node=heapq.heappop(candidates); worst=-best[0][0]  # 计算并保存当前步骤的中间状态。
            if len(best)>=ef and d>worst: break  # 按当前条件选择后续控制路径。
            for other in self.graph[level].get(node,set()):  # 遍历输入元素以累积或检查结果。
                if other in visited: continue  # 按当前条件选择后续控制路径。
                visited.add(other); od=self._d(q,other)  # 计算并保存当前步骤的中间状态。
                if len(best)<ef or od<-best[0][0]:  # 按当前条件选择后续控制路径。
                    heapq.heappush(candidates,(od,other)); heapq.heappush(best,(-od,other))  # 执行当前语句以推进本节示例。
                    if len(best)>ef: heapq.heappop(best)  # 按当前条件选择后续控制路径。
        return sorted([(-neg,n) for neg,n in best],key=lambda z:(z[0],z[1]))  # 返回当前分支计算出的结果。
probe69=HNSW69(); probe69.vectors={"a":np.array([0.,0.],np.float32),"b":np.array([1.,0.],np.float32),"c":np.array([2.,0.],np.float32)}; probe69.graph[0]["a"]={"b"}; probe69.graph[0]["b"]={"a","c"}; probe69.graph[0]["c"]={"b"}  # 计算并保存当前步骤的中间状态。
assert probe69.greedy(np.array([1.8,0.],np.float32),"a",0)=="c"  # 用受控断言验证关键不变量。
layer_probe69=probe69.search_layer(np.array([.9,0.],np.float32),["a"],3,0)  # 计算并保存当前步骤的中间状态。
assert [n for _,n in layer_probe69]==["b","a","c"] and len(layer_probe69)==3  # 用受控断言验证关键不变量。

## 4. 插入、双向连边与度数裁剪

插入从最高层 entry 开始贪心下降；在新节点存在的每层，用 `efConstruction` 搜候选，选择最近 `M` 个并双向连边。邻居超度后按距离裁剪，同时删除对应反向边，维持 reciprocity。

真正 HNSW 使用 diversity heuristic，避免所有边都指向同一方向的近邻；本例的 nearest-M 是教学简化，后续会在“生产差距”中明确。

In [ ]:
def _prune69(self,node,level):  # 定义本节可复用的核心函数。
    neighbors=self.graph[level][node]; ordered=sorted(neighbors,key=lambda n:(dist69(self.vectors[node],self.vectors[n]),n)); selected=[]  # 计算并保存当前步骤的中间状态。
    for candidate in ordered:  # 遍历输入元素以累积或检查结果。
        d_to_node=dist69(self.vectors[node],self.vectors[candidate])  # 计算并保存当前步骤的中间状态。
        if all(dist69(self.vectors[candidate],self.vectors[chosen])>=d_to_node for chosen in selected): selected.append(candidate)  # 按当前条件选择后续控制路径。
        if len(selected)==self.m: break  # 按当前条件选择后续控制路径。
    if len(selected)<self.m:  # 按当前条件选择后续控制路径。
        selected.extend(n for n in ordered if n not in selected and len(selected)<self.m)  # 执行当前语句以推进本节示例。
    keep=set(selected)  # 计算并保存当前步骤的中间状态。
    removed=set(neighbors)-keep; self.graph[level][node]=keep  # 计算并保存当前步骤的中间状态。
    for other in removed: self.graph[level][other].discard(node)  # 遍历输入元素以累积或检查结果。
def insert69(self,node,vector,level=None):  # 定义本节可复用的核心函数。
    vector=np.asarray(vector,np.float32)  # 计算并保存当前步骤的中间状态。
    if node in self.vectors or vector.ndim!=1 or vector.shape!=(2,) or not np.isfinite(vector).all(): raise ValueError("insert_contract")  # 按当前条件选择后续控制路径。
    level=self.sampler.sample() if level is None else int(level); self.vectors[node]=vector.copy(); self.levels[node]=level  # 计算并保存当前步骤的中间状态。
    if self.entry is None: self.entry=node; self.max_level=level; return  # 按当前条件选择后续控制路径。
    entry=self.entry  # 计算并保存当前步骤的中间状态。
    for lev in range(self.max_level,level,-1): entry=self.greedy(vector,entry,lev)  # 遍历输入元素以累积或检查结果。
    for lev in range(min(level,self.max_level),-1,-1):  # 遍历输入元素以累积或检查结果。
        candidates=self.search_layer(vector,[entry],self.ef_construction,lev); selected=[n for _,n in candidates if n!=node][:self.m]  # 计算并保存当前步骤的中间状态。
        for other in selected: self.graph[lev][node].add(other); self.graph[lev][other].add(node); _prune69(self,other,lev)  # 遍历输入元素以累积或检查结果。
        _prune69(self,node,lev)  # 执行当前语句以推进本节示例。
        if candidates: entry=candidates[0][1]  # 按当前条件选择后续控制路径。
    if level>self.max_level: self.entry=node; self.max_level=level  # 按当前条件选择后续控制路径。
HNSW69.insert=insert69  # 计算并保存当前步骤的中间状态。
small69=HNSW69(m=2,ef_construction=4,seed=1)  # 计算并保存当前步骤的中间状态。
small69.insert("a",[0,0],1); small69.insert("b",[1,0],0); small69.insert("c",[2,0],0)  # 执行当前语句以推进本节示例。
assert small69.entry=="a" and small69.max_level==1 and len(small69.vectors)==3  # 用受控断言验证关键不变量。
assert all(node in small69.graph[lvl][other] for lvl,rows in small69.graph.items() for node,ns in rows.items() for other in ns)  # 用受控断言验证关键不变量。
assert all(len(ns)<=small69.m for rows in small69.graph.values() for ns in rows.values())  # 用受控断言验证关键不变量。
try: small69.insert("a",[3,0]); raise AssertionError("duplicate id accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="insert_contract"  # 捕获预期异常并验证失败分支。

## 5. 构建完整图与结构不变量

插入顺序会影响图，因此 build seed、数据顺序和距离实现都属于版本。构建后检查：entry 在最高层、每条边双向、边两端都存在、节点只出现在不高于自身 level 的层、度数不超过 `M`。

生产构建常随机打散或批量优化；在线持续插入会积累质量差异，需要周期性 shadow rebuild。

In [ ]:
hnsw69=HNSW69(m=8,ef_construction=40,seed=6903)  # 计算并保存当前步骤的中间状态。
for node,vec in zip(ids69,data69): hnsw69.insert(node,vec)  # 遍历输入元素以累积或检查结果。
assert len(hnsw69.vectors)==210 and hnsw69.entry in hnsw69.vectors  # 用受控断言验证关键不变量。
assert hnsw69.levels[hnsw69.entry]==hnsw69.max_level  # 用受控断言验证关键不变量。
assert all(other in hnsw69.vectors for rows in hnsw69.graph.values() for ns in rows.values() for other in ns)  # 用受控断言验证关键不变量。
assert all(node in hnsw69.graph[level][other] for level,rows in hnsw69.graph.items() for node,ns in rows.items() for other in ns)  # 用受控断言验证关键不变量。
assert all(len(ns)<=hnsw69.m for rows in hnsw69.graph.values() for ns in rows.values())  # 用受控断言验证关键不变量。
assert all(level<=hnsw69.levels[node] for level,rows in hnsw69.graph.items() for node in rows)  # 用受控断言验证关键不变量。

## 6. 查询、稳定排序与软删除

查询从顶层 entry 贪心到第 1 层，再在第 0 层用 `efSearch` 展开，过滤 tombstone 后取前 k。若 entry 被删除仍可用于导航，但不能返回；若大量删除，图会变稀疏且距离计算浪费，需要重建。

相同距离必须有稳定 ID tie-break，否则多副本可能返回不同顺序，影响缓存与回归测试。

In [ ]:
def query_hnsw69(self,q,k=5,ef=30):  # 定义本节可复用的核心函数。
    q=np.asarray(q,np.float32)  # 计算并保存当前步骤的中间状态。
    if q.shape!=(2,) or k<1 or ef<k or self.entry is None: raise ValueError("query_contract")  # 按当前条件选择后续控制路径。
    entry=self.entry  # 计算并保存当前步骤的中间状态。
    for level in range(self.max_level,0,-1): entry=self.greedy(q,entry,level)  # 遍历输入元素以累积或检查结果。
    candidates=self.search_layer(q,[entry],ef,0); live=[(d,n) for d,n in candidates if n not in self.deleted]  # 计算并保存当前步骤的中间状态。
    return [(n,d) for d,n in sorted(live,key=lambda z:(z[0],z[1]))[:k]]  # 返回当前分支计算出的结果。
def delete_hnsw69(self,node):  # 定义本节可复用的核心函数。
    if node not in self.vectors: raise KeyError("unknown_node")  # 按当前条件选择后续控制路径。
    self.deleted.add(node)  # 执行当前语句以推进本节示例。
HNSW69.query=query_hnsw69; HNSW69.delete=delete_hnsw69  # 计算并保存当前步骤的中间状态。
result69=hnsw69.query(query69[0],5,30)  # 计算并保存当前步骤的中间状态。
assert len(result69)==5 and [d for _,d in result69]==sorted(d for _,d in result69)  # 用受控断言验证关键不变量。
victim69=result69[0][0]; hnsw69.delete(victim69); result_after69=hnsw69.query(query69[0],5,40)  # 计算并保存当前步骤的中间状态。
assert victim69 not in {n for n,_ in result_after69} and victim69 in hnsw69.deleted  # 用受控断言验证关键不变量。
try: hnsw69.query(query69[0],5,4); raise AssertionError("ef<k accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="query_contract"  # 捕获预期异常并验证失败分支。

## 7. Recall@K 与 `efSearch` 成本曲线

exact top-k 遍历所有 live 向量。比较 `ef=8/20/60` 的 Recall@5；距离计算次数在本实现未逐项计数，以 visited/ef 近似预算。注意单个 query 上 recall 不一定严格单调，但聚合通常随 ef 增长。

线上要按 query 难度、过滤条件和新旧 embedding 分片统计 recall，不应只报总体均值。

In [ ]:
def exact_hnsw69(q,k):  # 定义本节可复用的核心函数。
    rows=sorted([(dist69(q,v),n) for n,v in hnsw69.vectors.items() if n not in hnsw69.deleted],key=lambda z:(z[0],z[1]))  # 计算并保存当前步骤的中间状态。
    return [n for _,n in rows[:k]]  # 返回当前分支计算出的结果。
def recall_hnsw69(ef,k=5):  # 定义本节可复用的核心函数。
    vals=[]  # 计算并保存当前步骤的中间状态。
    for q in query69:  # 遍历输入元素以累积或检查结果。
        approx={n for n,_ in hnsw69.query(q,k,ef)}; vals.append(len(approx&set(exact_hnsw69(q,k)))/k)  # 计算并保存当前步骤的中间状态。
    return float(np.mean(vals))  # 返回当前分支计算出的结果。
rec8_69=recall_hnsw69(8); rec20_69=recall_hnsw69(20); rec60_69=recall_hnsw69(60)  # 计算并保存当前步骤的中间状态。
assert all(0<=x<=1 for x in (rec8_69,rec20_69,rec60_69))  # 用受控断言验证关键不变量。
assert rec60_69>=rec8_69 and rec60_69>.85  # 用受控断言验证关键不变量。
assert hnsw69.query(query69[1],5,30)==hnsw69.query(query69[1],5,30)  # 用受控断言验证关键不变量。

## 8. 重建、过滤与发布快照

HNSW 原生软删除不会修复导航图。删除比例、degree 分布、孤立节点、分层人口和 recall 漂移达到阈值时，应从 live vectors 构建新版本并灰度切换。metadata filter 若在 ANN 后执行可能候选不足；可按租户/类别分图，或扩大 ef 后过滤。

manifest 绑定向量摘要、level、每层邻接、entry、M/efConstruction、距离和删除集合；仅绑定原向量不足以证明图没被替换。

In [ ]:
def graph_digest69(index):  # 定义本节可复用的核心函数。
    rows={str(l):{n:sorted(ns) for n,ns in sorted(v.items())} for l,v in sorted(index.graph.items())}  # 计算并保存当前步骤的中间状态。
    vectors={n:np.ascontiguousarray(v).tobytes().hex() for n,v in sorted(index.vectors.items())}  # 计算并保存当前步骤的中间状态。
    payload={"vectors":vectors,"levels":index.levels,"graph":rows,"entry":index.entry,"max_level":index.max_level,"deleted":sorted(index.deleted)}  # 计算并保存当前步骤的中间状态。
    return sha69(canonical69(payload).encode())  # 返回当前分支计算出的结果。
manifest69={"artifact_id":"hnsw-demo-v1","metric":"squared_l2","dim":2,"M":hnsw69.m,"efConstruction":hnsw69.ef_construction,"graph_digest":graph_digest69(hnsw69),"build_seed":6903}  # 计算并保存当前步骤的中间状态。
TRUST69=MappingProxyType({manifest69["artifact_id"]:sha69(canonical69(manifest69).encode())})  # 计算并保存当前步骤的中间状态。
def load_graph69(index,m):  # 定义本节可复用的核心函数。
    actual=copy.deepcopy(m); actual["graph_digest"]=graph_digest69(index)  # 计算并保存当前步骤的中间状态。
    if TRUST69.get(actual.get("artifact_id"))!=sha69(canonical69(actual).encode()): raise RuntimeError("untrusted_hnsw_graph")  # 按当前条件选择后续控制路径。
    return MappingProxyType(actual)  # 返回当前分支计算出的结果。
pub69=load_graph69(hnsw69,manifest69)  # 计算并保存当前步骤的中间状态。
assert pub69["M"]==8 and isinstance(TRUST69,MappingProxyType)  # 用受控断言验证关键不变量。
forged_hnsw69=copy.deepcopy(manifest69); forged_hnsw69["M"]=99  # 计算并保存当前步骤的中间状态。
try: load_graph69(hnsw69,forged_hnsw69); raise AssertionError("forged HNSW accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="untrusted_hnsw_graph"  # 捕获预期异常并验证失败分支。
print({"nodes":len(hnsw69.vectors),"max_level":hnsw69.max_level,"recall@ef8":round(rec8_69,3),"recall@ef60":round(rec60_69,3)})  # 执行当前语句以推进本节示例。

## 9. 复杂度、生产差距与来源

理想查询近似对数导航加 `efSearch` 局部扩展，但最坏情况仍可能接近线性。内存主要是原向量和约 `O(NM)` 邻接。nearest-M 简化会降低多样性；生产实现还需要 heuristic neighbor selection、锁、prefetch、SIMD、压缩、批查询和故障恢复。

- Malkov & Yashunin, [Efficient and robust approximate nearest neighbor search using HNSW](https://arxiv.org/abs/1603.09320), TPAMI 2020。
- hnswlib, [Algorithm parameters and implementation reference](https://github.com/nmslib/hnswlib)。
- Aumüller et al., [ANN-Benchmarks](https://arxiv.org/abs/1807.05614)，评估方法背景。